# Qwen 3.5 Architecture Tutorial

**Comprehensive guide to understanding Qwen 3.5's hybrid GDN architecture**

## What is Qwen 3.5?

Qwen 3.5 uses a **hybrid architecture** combining:
- **75% GDN (Gated Differentiable Network) layers** - Linear recurrent processing
- **25% Attention layers** - Full self-attention every 4th layer

This 3:1 ratio provides:
- ✓ Efficient long-context processing (GDN is O(n) vs attention's O(n²))
- ✓ Strong reasoning capabilities (periodic attention for global context)
- ✓ Reduced memory footprint

## Learning Objectives

1. **Model Structure** - Configuration, layer types, parameter counts
2. **Layer-by-Layer Analysis** - GDN vs Attention layer patterns
3. **Hidden State Extraction** - Forward hooks to capture internal representations
4. **Logit Lens** - Visualize what model "thinks" at each layer
5. **Conflict Resolution** - How layers resolve parametric vs in-context knowledge

## References

- [Qwen 3.5 Model Card](https://huggingface.co/Qwen/Qwen3.5-4B)
- [GDN Paper (Lightning Attention)](https://arxiv.org/abs/2401.04695)
- [Logit Lens (nostalgebraist)](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)

## Section 1: Setup

In [ ]:
# Cell 1.1: Imports
import sys
from pathlib import Path
import json

import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import psutil

# Setup matplotlib
%matplotlib inline
sns.set_style("whitegrid")

# Add project to path
sys.path.insert(0, '/Users/smaller225/code/Hybrid_fasteval/project')

# Import project modules
from models.load_model import load_model
from models.layer_utils import get_layer_type_map, inspect_layer_types
from utils.notebook_helpers import (
    setup_project_path,
    format_model_info,
    extract_hidden_states,
    compute_logit_lens,
    get_final_norm,
    get_lm_head
)

# Create output directory
OUTPUT_DIR = Path('/Users/smaller225/code/Hybrid_fasteval/results/notebook_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Imports successful")

In [ ]:
# Cell 1.2: Configuration
MODEL_NAME = 'qwen3.5-4b'  # Options: qwen3.5-4b, qwen3.5-2b
DEVICE = 'cpu'  # Mac CPU only

# Check available memory
available_memory_gb = psutil.virtual_memory().available / 1e9
print(f"Available RAM: {available_memory_gb:.1f} GB")
if available_memory_gb < 10:
    print("⚠️  Low memory! Consider using 'qwen3.5-2b' instead")

print(f"\nModel: {MODEL_NAME}")
print(f"Device: {DEVICE}")

In [ ]:
# Cell 1.3: Load Model
print(f"Loading model: {MODEL_NAME}...")
print("(This may take 30-60 seconds on CPU)\n")

model, tokenizer = load_model(MODEL_NAME, device_map=DEVICE)

# Display model info
model_info = format_model_info(model, tokenizer)
print("\n✓ Model loaded successfully!\n")
print(model_info.to_string(index=False))

## Section 2: Model Configuration

In [ ]:
# Cell 2.1: Configuration Details
config = model.config

config_data = {
    'Property': [
        'model_type',
        'hidden_size',
        'num_hidden_layers',
        'num_attention_heads',
        'vocab_size',
        'max_position_embeddings',
    ],
    'Value': [
        config.model_type,
        config.hidden_size,
        config.num_hidden_layers,
        config.num_attention_heads,
        config.vocab_size,
        getattr(config, 'max_position_embeddings', 'N/A'),
    ]
}

df_config = pd.DataFrame(config_data)
print("Model Configuration:")
print(df_config.to_string(index=False))

In [ ]:
# Cell 2.2: Layer Type Mapping
layer_map = get_layer_type_map(MODEL_NAME, model=model)

total_layers = layer_map['total_layers']
n_attn = len(layer_map['attention_indices'])
n_linear = len(layer_map['linear_indices'])

print(f"Layer Type Distribution:")
print(f"{'='*60}")
print(f"Total Layers: {total_layers}")
print(f"Attention Layers: {n_attn} ({n_attn/total_layers*100:.1f}%)")
print(f"GDN/Linear Layers: {n_linear} ({n_linear/total_layers*100:.1f}%)")
print(f"{'='*60}\n")

print(f"Attention Layer Positions: {layer_map['attention_indices']}")
print(f"\nPattern: Every {total_layers // n_attn}th layer is attention (3:1 ratio)")

In [ ]:
# Cell 2.3: Layer-by-Layer Breakdown
print(f"\nLayer-by-Layer Breakdown:")
print(f"{'='*80}")
print(f"{'Layer':<8} {'Type':<12} {'Module Name':<30} {'Module Class':<25}")
print(f"{'='*80}")

# Access layers
layers = model.model.layers

for idx, layer in enumerate(layers):
    layer_type = 'Attention' if idx in layer_map['attention_indices'] else 'GDN/Linear'
    module_name = f"model.layers[{idx}]"
    module_class = layer.__class__.__name__
    marker = "  <-- ATTN" if idx in layer_map['attention_indices'] else ""
    
    print(f"{idx:<8} {layer_type:<12} {module_name:<30} {module_class:<25}{marker}")

print(f"{'='*80}")

In [ ]:
# Cell 2.4: Visualize Layer Pattern
fig, ax = plt.subplots(figsize=(12, 4))

# Create color mapping
colors = ['blue' if i in layer_map['attention_indices'] else 'red' 
          for i in range(total_layers)]

# Bar chart
ax.bar(range(total_layers), [1]*total_layers, color=colors, alpha=0.7, edgecolor='black')
ax.set_xlabel('Layer Index', fontsize=12)
ax.set_ylabel('Layer Type', fontsize=12)
ax.set_title(f'{MODEL_NAME} Hybrid Architecture Pattern (GDN 3:1)', fontsize=14, fontweight='bold')
ax.set_yticks([0, 1])
ax.set_yticklabels(['', ''])
ax.set_xticks(range(total_layers))

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='blue', alpha=0.7, label='Attention'),
    Patch(facecolor='red', alpha=0.7, label='GDN/Linear')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()

# Save
output_path = OUTPUT_DIR / f"qwen_layer_pattern_{MODEL_NAME}.png"
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved to: {output_path}")

plt.show()

## Section 3: Module Inspection

In [ ]:
# Cell 3.1: Inspect Single Layer
INSPECT_LAYER = 0  # Change to inspect different layers

layer = model.model.layers[INSPECT_LAYER]
print(f"Layer {INSPECT_LAYER} Submodules:")
print(f"{'='*80}")

for name, module in layer.named_modules():
    if name == '':
        continue
    n_params = sum(p.numel() for p in module.parameters())
    print(f"{name:<40} {module.__class__.__name__:<30} {n_params:>10,} params")

print(f"{'='*80}")

In [ ]:
# Cell 3.2: Attention Layer Detail
if len(layer_map['attention_indices']) > 0:
    attn_layer_idx = layer_map['attention_indices'][0]
    attn_layer = model.model.layers[attn_layer_idx]
    
    print(f"Attention Layer {attn_layer_idx} Details:")
    print(f"{'='*60}")
    
    # Get attention module (structure may vary)
    if hasattr(attn_layer, 'self_attn'):
        attn = attn_layer.self_attn
        print(f"Hidden Size: {config.hidden_size}")
        print(f"Num Heads: {config.num_attention_heads}")
        print(f"Head Dim: {config.hidden_size // config.num_attention_heads}")
        print(f"\nSubmodules:")
        for name, module in attn.named_children():
            print(f"  {name}: {module.__class__.__name__}")
    else:
        print("Attention structure varies - inspect manually")
        print(f"Available attributes: {dir(attn_layer)[:10]}...")
    
    print(f"{'='*60}")
else:
    print("No attention layers found")

In [ ]:
# Cell 3.3: GDN Layer Detail
if len(layer_map['linear_indices']) > 0:
    gdn_layer_idx = layer_map['linear_indices'][0]
    gdn_layer = model.model.layers[gdn_layer_idx]
    
    print(f"GDN/Linear Layer {gdn_layer_idx} Details:")
    print(f"{'='*60}")
    print(f"GDN layers process recurrently without full attention")
    print(f"This enables O(n) complexity vs O(n²) for attention\n")
    
    print(f"Submodules:")
    for name, module in gdn_layer.named_children():
        print(f"  {name}: {module.__class__.__name__}")
    
    print(f"{'='*60}")
else:
    print("No GDN/Linear layers found")

## Section 4: Hidden States Extraction

In [ ]:
# Cell 4.1: Forward Pass Setup
PROMPT = "The capital of France is"

# Tokenize
inputs = tokenizer(PROMPT, return_tensors='pt').to(DEVICE)
token_ids = inputs['input_ids'][0]
tokens = tokenizer.convert_ids_to_tokens(token_ids)

print(f"Prompt: {PROMPT}")
print(f"\nTokens: {tokens}")
print(f"Token IDs: {token_ids.tolist()}")
print(f"Sequence Length: {len(token_ids)}")

In [ ]:
# Cell 4.2: Extract Hidden States
print("Extracting hidden states from all layers...\n")

hidden_states_dict = extract_hidden_states(
    model,
    inputs,
    layer_module_path="model.layers"
)

print(f"✓ Extracted hidden states from {len(hidden_states_dict)} layers")
print(f"\nShape: (batch_size, seq_len, hidden_dim)")
print(f"Example (Layer 0): {hidden_states_dict[0].shape}")

In [ ]:
# Cell 4.3: Analyze Hidden States
analysis = []

for layer_idx, hidden_states in hidden_states_dict.items():
    # Get last token representation
    last_token = hidden_states[0, -1, :]
    
    analysis.append({
        'Layer': layer_idx,
        'Type': 'Attention' if layer_idx in layer_map['attention_indices'] else 'GDN',
        'Mean': last_token.mean().item(),
        'Std': last_token.std().item(),
        'Min': last_token.min().item(),
        'Max': last_token.max().item(),
        'L2 Norm': torch.norm(last_token, p=2).item(),
    })

df_analysis = pd.DataFrame(analysis)
print("Hidden State Analysis (Last Token):")
print(df_analysis.to_string(index=False))

In [ ]:
# Cell 4.4: Visualize Evolution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Mean activation
colors_1 = ['blue' if t == 'Attention' else 'red' for t in df_analysis['Type']]
axes[0, 0].bar(df_analysis['Layer'], df_analysis['Mean'], color=colors_1, alpha=0.7)
axes[0, 0].set_title('Mean Activation per Layer')
axes[0, 0].set_xlabel('Layer')
axes[0, 0].set_ylabel('Mean')
axes[0, 0].axhline(y=0, color='black', linestyle='--', alpha=0.3)

# Plot 2: Std deviation
axes[0, 1].bar(df_analysis['Layer'], df_analysis['Std'], color=colors_1, alpha=0.7)
axes[0, 1].set_title('Std Deviation per Layer')
axes[0, 1].set_xlabel('Layer')
axes[0, 1].set_ylabel('Std')

# Plot 3: L2 norm
axes[1, 0].plot(df_analysis['Layer'], df_analysis['L2 Norm'], marker='o', linewidth=2)
# Highlight attention layers
attn_layers = df_analysis[df_analysis['Type'] == 'Attention']
axes[1, 0].scatter(attn_layers['Layer'], attn_layers['L2 Norm'], 
                   color='blue', s=100, zorder=5, label='Attention')
axes[1, 0].set_title('L2 Norm per Layer')
axes[1, 0].set_xlabel('Layer')
axes[1, 0].set_ylabel('L2 Norm')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Range (max-min)
df_analysis['Range'] = df_analysis['Max'] - df_analysis['Min']
axes[1, 1].bar(df_analysis['Layer'], df_analysis['Range'], color=colors_1, alpha=0.7)
axes[1, 1].set_title('Range (Max - Min) per Layer')
axes[1, 1].set_xlabel('Layer')
axes[1, 1].set_ylabel('Range')

plt.tight_layout()

# Save
output_path = OUTPUT_DIR / f"qwen_hidden_states_{MODEL_NAME}.png"
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Saved to: {output_path}")

plt.show()

## Section 5: Logit Lens

**What is Logit Lens?**

Project each layer's hidden state through the final normalization and language model head to see what the model "thinks" at each layer. This reveals:
- How predictions evolve across layers
- When the model "decides" on an answer
- Differences between attention and GDN layers

In [ ]:
# Cell 5.1: Setup Logit Lens
final_norm = get_final_norm(model)
lm_head = get_lm_head(model)

print(f"Final Norm: {final_norm.__class__.__name__}")
print(f"LM Head: {lm_head.__class__.__name__}")
print(f"✓ Logit lens components ready")

In [ ]:
# Cell 5.2: Compute Layer-wise Predictions
logit_lens_results = []

for layer_idx, hidden_states in hidden_states_dict.items():
    logits, top_k_ids, top_k_probs = compute_logit_lens(
        hidden_states,
        final_norm,
        lm_head,
        device=DEVICE,
        top_k=5
    )
    
    # Decode top tokens
    top_tokens = [tokenizer.decode([tid]) for tid in top_k_ids]
    
    logit_lens_results.append({
        'Layer': layer_idx,
        'Type': 'Attention' if layer_idx in layer_map['attention_indices'] else 'GDN',
        'Top-1 Token': top_tokens[0],
        'Top-1 Prob': top_k_probs[0],
        'Top-5 Tokens': ', '.join(top_tokens),
        'Top-5 Probs': [f"{p:.3f}" for p in top_k_probs],
    })

df_logit_lens = pd.DataFrame(logit_lens_results)
print("✓ Computed logit lens for all layers")

In [ ]:
# Cell 5.3: Display Results
print(f"\nLogit Lens Results for: '{PROMPT}'")
print(f"{'='*100}")
print(df_logit_lens[['Layer', 'Type', 'Top-1 Token', 'Top-1 Prob']].to_string(index=False))
print(f"{'='*100}")

In [ ]:
# Cell 5.4: Visualize Logit Lens
fig, ax = plt.subplots(figsize=(12, 6))

# Line plot with colors by layer type
for layer_type, color in [('Attention', 'blue'), ('GDN', 'red')]:
    df_type = df_logit_lens[df_logit_lens['Type'] == layer_type]
    ax.plot(df_type['Layer'], df_type['Top-1 Prob'], 
            marker='o', color=color, label=layer_type, linewidth=2, markersize=8)

# Mark attention layers with vertical lines
for attn_idx in layer_map['attention_indices']:
    ax.axvline(x=attn_idx, color='blue', alpha=0.2, linestyle='--')

ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Top-1 Probability', fontsize=12)
ax.set_title(f'Logit Lens: Prediction Confidence Across Layers\nPrompt: "{PROMPT}"', 
             fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()

# Save
output_path = OUTPUT_DIR / f"qwen_logit_lens_{MODEL_NAME}.png"
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved to: {output_path}")

plt.show()

## Section 6: Conflict Example

**Test how layers resolve conflict between parametric and in-context knowledge**

In [ ]:
# Cell 6.1: Setup Conflict Prompt
CONFLICT_PROMPT = """Context: According to recent records, Paris is associated with Germany.

Q: What is Paris associated with?
A:"""

NO_CONTEXT_PROMPT = """Q: What is Paris associated with?
A:"""

print("Conflict Scenario:")
print(f"{'='*60}")
print(f"Parametric Knowledge: Paris → France")
print(f"In-context Information: Paris → Germany")
print(f"{'='*60}\n")
print(f"Conflict Prompt:\n{CONFLICT_PROMPT}")

In [ ]:
# Cell 6.2: Extract Conflict Hidden States
conflict_inputs = tokenizer(CONFLICT_PROMPT, return_tensors='pt').to(DEVICE)

print("Extracting hidden states for conflict prompt...\n")

conflict_hs_dict = extract_hidden_states(
    model,
    conflict_inputs,
    layer_module_path="model.layers"
)

print(f"✓ Extracted hidden states from {len(conflict_hs_dict)} layers")

In [ ]:
# Cell 6.3: Conflict Logit Lens
# Get token IDs for France and Germany
france_id = tokenizer.encode('France', add_special_tokens=False)[0]
germany_id = tokenizer.encode('Germany', add_special_tokens=False)[0]

print(f"Token IDs: France={france_id}, Germany={germany_id}\n")

conflict_results = []

for layer_idx, hidden_states in conflict_hs_dict.items():
    # Get last token
    last_hidden = hidden_states[:, -1, :].to(DEVICE)
    
    # Apply final norm and project
    normed = final_norm(last_hidden)
    logits = lm_head(normed)
    probs = F.softmax(logits, dim=-1)
    
    # Extract probabilities for France and Germany
    prob_france = probs[0, france_id].item()
    prob_germany = probs[0, germany_id].item()
    
    # Get top prediction
    top_prob, top_id = torch.max(probs[0], dim=0)
    top_token = tokenizer.decode([top_id.item()])
    
    conflict_results.append({
        'Layer': layer_idx,
        'Type': 'Attention' if layer_idx in layer_map['attention_indices'] else 'GDN',
        'P(France)': prob_france,
        'P(Germany)': prob_germany,
        'Dominant': 'France' if prob_france > prob_germany else 'Germany',
        'Top Token': top_token,
    })

df_conflict = pd.DataFrame(conflict_results)
print("✓ Computed conflict resolution across layers")
print("\nConflict Resolution:")
print(df_conflict[['Layer', 'Type', 'P(France)', 'P(Germany)', 'Dominant']].to_string(index=False))

In [ ]:
# Cell 6.4: Visualize Conflict Resolution
fig, ax = plt.subplots(figsize=(12, 6))

# Plot both probabilities
ax.plot(df_conflict['Layer'], df_conflict['P(France)'], 
        marker='o', color='blue', label='P(France) - Parametric', linewidth=2, markersize=8)
ax.plot(df_conflict['Layer'], df_conflict['P(Germany)'], 
        marker='s', color='red', label='P(Germany) - In-context', linewidth=2, markersize=8)

# Shade attention layers
for attn_idx in layer_map['attention_indices']:
    ax.axvspan(attn_idx - 0.3, attn_idx + 0.3, alpha=0.2, color='gray', label='Attention' if attn_idx == layer_map['attention_indices'][0] else '')

ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Conflict Resolution: Parametric vs In-context Knowledge\nPrompt: "Paris is associated with Germany"', 
             fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, max(df_conflict['P(France)'].max(), df_conflict['P(Germany)'].max()) * 1.1])

plt.tight_layout()

# Save
output_path = OUTPUT_DIR / f"qwen_conflict_resolution_{MODEL_NAME}.png"
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Saved to: {output_path}")

plt.show()

# Print dominant prediction per layer
print("\nLayer-wise Dominant Prediction:")
for _, row in df_conflict.iterrows():
    print(f"  Layer {row['Layer']:2d} ({row['Type']:9s}): {row['Dominant']}")

## Section 7: Summary

### Key Insights

From this tutorial, we learned:

1. **Hybrid Architecture**
   - Qwen 3.5 uses 3:1 ratio (75% GDN, 25% attention)
   - Attention layers appear every 4th layer
   - This balances efficiency (GDN) with reasoning (attention)

2. **Layer Behavior**
   - Hidden states evolve progressively across layers
   - Attention layers may show distinct patterns in L2 norm and activation statistics
   - GDN layers process efficiently without quadratic complexity

3. **Logit Lens Observations**
   - Early layers have lower prediction confidence
   - Later layers converge to final answer
   - Top-1 probability increases as depth increases

4. **Conflict Resolution**
   - Can track how parametric vs in-context knowledge competes
   - Different layers may favor different answers
   - Final output is determined by last layer's probabilities

### Applications

- **Debugging**: Identify where model makes mistakes
- **Interpretability**: Understand internal reasoning process
- **Research**: Study knowledge representation across layers

### Next Steps

**Further Exploration:**
1. Compare different prompts (factual, reasoning, creative)
2. Analyze longer contexts to see GDN efficiency gains
3. Test different conflict scenarios
4. Compare Qwen 3.5 with pure attention models

**Related Notebooks:**
- `debug_conflict_responses.ipynb` - Debug conflict dataset results
- Experimental results in `/results/` directory

**Resources:**
- [Qwen Technical Report](https://arxiv.org/abs/2309.16609)
- [Flash Linear Attention](https://github.com/fla-org/flash-linear-attention)
- [Logit Lens Analysis](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)